# Constituer un jeu d'appels d'outils pour l'agent du service client

Le fil rouge de la formation : l'agent du service client de Biclou & Cie, qui lit le message d'un client, appelle les outils du magasin (la fiche d'une commande, le stock, les règles du magasin, un ticket, le transfert à un conseiller), puis répond selon la charte. Avec un petit modèle de base, il se trompe : il répond sans chercher, choisit le mauvais outil, invente des arguments. Ce TP constitue, à partir de ces erreurs, le jeu d'entraînement qui doit l'en corriger : le TP suivant y affinera Qwen3-1.7B par QLoRA, et les derniers chapitres compareront les deux modèles sur un jeu d'évaluation figé.

1. Les outils en Python
2. Le format conversationnel
3. Les erreurs de l'agent de référence
4. Des erreurs aux exemples
5. Des exemples synthétiques
6. Filtrer les exemples synthétiques
7. Équilibrer, découper, enregistrer

Ce notebook demande un GPU : *Exécution > Modifier le type d'exécution > GPU T4*, sur lequel il tient en une demi-heure environ (une estimation). Sans GPU, il tourne quand même, en plus petit : Qwen3-0.6B pour l'agent au lieu de Qwen3-1.7B, sur 12 demandes au lieu de 240, et Qwen3-1.7B pour écrire 4 messages synthétiques au lieu de Qwen3-4B-Instruct-2507 pour près de 300 : 20 minutes environ sur deux processeurs, comme ceux de Colab. Le jeu obtenu manque alors des cas que l'historique a peu : le TP suivant part d'un jeu de référence, construit par ce même code sur GPU.

## Données et modèles

Les données de l'agent, dans le dossier `agent/` du jeu `bike-shop-support` : `demandes.jsonl`, l'historique annoté, 1 300 demandes de clients avec les appels que l'agent devait faire, leurs résultats et la réponse, d'où viennent tous les exemples ; `eval.jsonl`, le jeu d'évaluation figé, 139 demandes sur lesquelles les chapitres suivants compareront les modèles, dont on ne se sert ici que pour vérifier que rien ne s'en approche ; `commandes.csv` et `stock.csv`, ce que lisent les outils ; `outils.json`, leurs schémas de référence. Et `charte.md`, la charte du service client, dont la partie « Politique » est ce que cherche l'outil `chercher_politique`.

In [ ]:
!wget -q https://github.com/shuuchuu/datasets/raw/refs/heads/main/bike-shop-support/agent/demandes.jsonl
!wget -q https://github.com/shuuchuu/datasets/raw/refs/heads/main/bike-shop-support/agent/eval.jsonl
!wget -q https://github.com/shuuchuu/datasets/raw/refs/heads/main/bike-shop-support/agent/commandes.csv
!wget -q https://github.com/shuuchuu/datasets/raw/refs/heads/main/bike-shop-support/agent/stock.csv
!wget -q https://github.com/shuuchuu/datasets/raw/refs/heads/main/bike-shop-support/agent/outils.json
!wget -q https://github.com/shuuchuu/datasets/raw/refs/heads/main/bike-shop-support/charte.md
%pip install -q datasets

In [ ]:
import collections
import copy
import gc
import json
import os
import random
import re
import time
import unicodedata
from typing import Literal

import pandas as pd
import torch
from datasets import load_dataset
from transformers import AutoModelForCausalLM, AutoTokenizer, DynamicCache
from transformers.utils import get_json_schema

GPU = torch.cuda.is_available()
DTYPE = torch.float16 if GPU else torch.float32  # sur CPU, float32 : bien plus rapide
DEVICE = "cuda" if GPU else "cpu"
MODELE_AGENT = "Qwen/Qwen3-1.7B" if GPU else "Qwen/Qwen3-0.6B"
MODELE_GENERALISTE = "Qwen/Qwen3-4B-Instruct-2507" if GPU else "Qwen/Qwen3-1.7B"
# Sans GPU, en plus petit : les demandes par comportement attendu que l'agent de
# référence traite (exercice 3), et les exemples synthétiques (exercice 5).
N_PAR_CLASSE = 20 if GPU else 1
N_SYNTHETIQUES = None if GPU else 4  # None : de quoi combler tout le déficit
print(MODELE_AGENT, "sur GPU" if GPU else "sans GPU, en plus petit")

demandes = pd.read_json("demandes.jsonl", lines=True, convert_dates=False)
eval_fige = pd.read_json("eval.jsonl", lines=True, convert_dates=False)
commandes = pd.read_csv("commandes.csv", dtype=str, keep_default_na=False)
commandes = commandes.set_index("numero")
stock = pd.read_csv("stock.csv", keep_default_na=False)
with open("outils.json", encoding="utf-8") as f:
  outils_reference = json.load(f)
with open("charte.md", encoding="utf-8") as f:
  charte = f.read()
tokenizer = AutoTokenizer.from_pretrained(MODELE_AGENT)
tokenizer.padding_side = "left"
print(len(demandes), "demandes dans l'historique,", len(eval_fige), "dans l'évaluation")


def libere() -> None:
  """Rend au GPU la mémoire des objets supprimés par `del`."""
  gc.collect()
  if GPU:
    torch.cuda.empty_cache()


def classe(demande) -> str:
  """Le comportement attendu : les outils, dans l'ordre, ou le cas sans outil."""
  noms = [a["name"] for a in demande["appels_attendus"]]
  if noms:
    return "+".join(noms)
  cas = "sans_numero" if demande["cas"] == "sans_numero" else demande["scenario"]
  return f"aucun:{cas}"


demandes["classe"] = demandes.apply(classe, axis=1)
demandes["classe"].value_counts()

La consigne de l'agent, la même pour le modèle de base et pour le modèle affiné : son rôle, quand appeler chaque outil, et les règles de réponse de la charte.

In [ ]:
CONSIGNES_AGENT = """Tu es l'agent du service client de Biclou & Cie, qui vend, répare et loue des vélos en ligne et dans six magasins (Lille, Amiens, Nice, Marseille, Nantes, Rennes). Pour chaque message de client, appelle les outils utiles, un à la fois, puis réponds au client.

Quand appeler un outil :
- un numéro de commande (CMD- et 6 chiffres) dans le message : chercher_commande d'abord ;
- une question sur les règles du magasin (horaires, tailles, garantie, atelier, retours, livraison, location, factures, compte) : chercher_politique ;
- la disponibilité d'un vélo dans un magasin : verifier_stock ;
- un problème que l'équipe doit suivre (colis abîmé, panne, réglage, réduction oubliée, suppression de compte) : creer_ticket ;
- la sécurité (freins, batterie), un colis perdu, un double prélèvement, un accident ou un vol pendant une location : escalader.
Sans outil : un suivi de commande sans son numéro (demande-le), un remerciement, une question sans rapport avec le magasin, une demande à refuser.

Ta réponse suit la charte : vouvoiement, « Bonjour, », ou « Bonjour Madame » ou « Bonjour Monsieur » et le nom de famille qui signe le message, 40 à 120 mots, une prochaine étape claire, aucun remboursement promis avant vérification, « Bonne journée, » puis « L'équipe Biclou & Cie » sur sa propre ligne, et en anglais pour un message en anglais."""

## 1. Les outils en Python

Pour un modèle, un outil est un schéma JSON : son nom, sa description, ses arguments et leurs types. `transformers` le tire d'une fonction Python, de ses annotations de type et de sa docstring au format Google (`get_json_schema`) : la même fonction est l'outil qu'on exécute et sa documentation pour le modèle. Une liste fermée de valeurs s'annote `Literal[...]` et devient une énumération (`enum`) du schéma, que le modèle voit et qu'on peut vérifier.

Trois des cinq outils sont donnés (cellule ci-dessous) : `chercher_politique`, une recherche par mots-clés dans les règles du magasin, la recherche documentaire exposée comme un outil ; `creer_ticket` ; et `escalader`. Écrivez les deux autres, sur les données téléchargées :

- `chercher_commande(numero)` : la fiche de la commande `numero` dans `commandes` (indexé par numéro), le numéro compris, sans ses champs vides, `montant` en nombre décimal et `paiements` en entier ; une erreur (`{"erreur": ...}`) pour un numéro qui n'est pas `CMD-` suivi de 6 chiffres, ou qui n'existe pas ;
- `verifier_stock(modele, taille, magasin)` : la ligne de `stock`, en un dictionnaire `modele`, `taille`, `magasin`, `quantite` (un entier), plus `reassort` quand la quantité est nulle ; une erreur si la ligne n'existe pas.

Leurs descriptions sont celles de `outils.json` : « Cherche une commande dans le système de commandes. », l'argument « le numéro de commande, CMD- suivi de 6 chiffres » et le retour « la fiche de la commande, ou une erreur si elle est introuvable » ; « Vérifie le stock d'un modèle de vélo dans un magasin. », les arguments « le modèle de vélo », « la taille de cadre » et « la ville du magasin », le retour « la quantité en stock et, en rupture, la date du prochain réassort ». Les valeurs possibles de `modele`, `taille` et `magasin` sont dans `MODELES`, `TAILLES` et `MAGASINS`.

Rangez ensuite les cinq fonctions dans le dictionnaire `FONCTIONS` (nom, fonction), et leurs schémas dans la liste `OUTILS`, dans cet ordre : `chercher_commande`, `verifier_stock`, `chercher_politique`, `creer_ticket`, `escalader`. Les schémas doivent être identiques à `outils_reference`, et vos fonctions rendre, sur les appels de l'historique, les résultats qu'il a enregistrés.

**Documentation utile**

- [`get_json_schema`](https://huggingface.co/docs/transformers/en/chat_extras#json-schemas)
- [`typing.Literal`](https://docs.python.org/3/library/typing.html#typing.Literal)
- [`DataFrame.loc`](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.loc.html)

In [ ]:
CATEGORIES = [
  "information",
  "livraison",
  "reparation",
  "retour",
  "location",
  "facturation",
  "compte",
]
PRIORITES = ["basse", "normale", "haute"]
RAISONS = ["securite", "colis_perdu", "paiement", "accident_location", "vol_location"]
MODELES = list(dict.fromkeys(stock["modele"]))
TAILLES = list(dict.fromkeys(stock["taille"]))
MAGASINS = list(dict.fromkeys(stock["magasin"]))
RUBRIQUES = {
  "Retours": "retours",
  "Livraison": "livraison",
  "Garantie": "garantie",
  "Atelier": "atelier",
  "Location": "location",
  "Facturation": "facturation",
  "Compte": "compte",
  "Magasins": "magasins",
  "Tailles de cadre": "tailles",
}


def lit_politique(charte: str) -> dict[str, list[str]]:
  """Les phrases de chaque rubrique de la partie « Politique » de la charte."""
  texte = charte.replace("\xa0", " ").split("## Politique", 1)[1]
  puces = re.findall(r"^- \*\*(.+?)\*\* : (.+?)(?=^- |\Z)", texte, re.M | re.S)
  return {
    RUBRIQUES[titre]: re.split(
      r"(?<=\.) (?=[A-ZÀ-Ý])", f"{titre} : {' '.join(corps.split())}"
    )
    for titre, corps in puces
  }


PASSAGES = lit_politique(charte)
THEMES = list(PASSAGES)


def racines(texte: str) -> set[str]:
  """Les mots de 4 lettres ou plus, comparés par leurs 5 premières lettres."""
  texte = unicodedata.normalize("NFKD", texte.lower())
  texte = "".join(c for c in texte if not unicodedata.combining(c))
  return {mot[:5] for mot in re.findall(r"[a-z]+", texte) if len(mot) >= 4}


def chercher_politique(theme: Literal[tuple(THEMES)], question: str) -> dict:
  """Cherche dans les règles du magasin les passages qui répondent à une question.

  Args:
    theme: la rubrique des règles où chercher
    question: la question, en quelques mots

  Returns:
    les passages de la rubrique les plus proches de la question
  """
  if theme not in PASSAGES:
    return {"erreur": f"rubrique inconnue : {theme}"}
  mots = racines(question)
  tries = sorted(
    enumerate(PASSAGES[theme]), key=lambda p: (-len(mots & racines(p[1])), p[0])
  )
  return {"theme": theme, "passages": [passage for _, passage in tries[:3]]}


compteurs = {"ticket": 30000, "dossier": 5000}
EQUIPES = {
  "securite": ("atelier", "aujourd'hui"),
  "colis_perdu": ("livraison", "sous 24 h"),
  "paiement": ("comptabilite", "sous 24 h"),
  "accident_location": ("location", "sous 24 h"),
  "vol_location": ("location", "sous 24 h"),
}


def creer_ticket(
  categorie: Literal[tuple(CATEGORIES)],
  priorite: Literal[tuple(PRIORITES)],
  resume: str,
) -> dict:
  """Crée un ticket pour une demande que l'équipe doit traiter.

  Args:
    categorie: la catégorie de la demande
    priorite: la priorité de la demande
    resume: la demande en une phrase

  Returns:
    le numéro du ticket créé
  """
  if categorie not in CATEGORIES or priorite not in PRIORITES:
    return {"erreur": "catégorie ou priorité inconnue"}
  compteurs["ticket"] += 1
  return {"ticket": f"T-{compteurs['ticket']}", "statut": "ouvert"}


def escalader(raison: Literal[tuple(RAISONS)], resume: str) -> dict:
  """Transmet la demande à un conseiller, pour un problème de sécurité ou d'argent.

  Args:
    raison: la raison du transfert
    resume: la situation en une phrase

  Returns:
    le numéro du dossier et l'équipe qui le prend en charge
  """
  if raison not in EQUIPES:
    return {"erreur": f"raison inconnue : {raison}"}
  compteurs["dossier"] += 1
  equipe, recontact = EQUIPES[raison]
  return {
    "dossier": f"ESC-{compteurs['dossier']}",
    "equipe": equipe,
    "recontact": recontact,
  }


print(chercher_politique("magasins", "le magasin est-il ouvert le lundi ?"))

In [ ]:
# Votre code ici

### Conseils supplémentaires

- La docstring au format Google : la description, une ligne vide, `Args:` et une ligne `nom: description` par argument, puis `Returns:` et la description du retour sur une seule ligne (`get_json_schema` garde les retours à la ligne d'une description) ; `chercher_politique` en est un exemple
- `Literal[tuple(MODELES)]` : l'énumération des valeurs de `MODELES`
- `numero in commandes.index` dit si la commande existe ; `commandes.loc[numero].to_dict()` donne sa ligne en dictionnaire, sans le numéro, qui est l'index ; les cellules vides y sont des chaînes vides
- `stock[(stock["modele"] == modele) & (stock["taille"] == taille) & (stock["magasin"] == magasin)]` : la ligne, ou un tableau vide (`.empty`)
- Exemple à vérifier : `verifier_stock("Biclou Urbain 3", "S", "Lille")` donne `{'modele': 'Biclou Urbain 3', 'taille': 'S', 'magasin': 'Lille', 'quantite': 2}`

```python
def chercher_commande(numero: str) -> dict:
  """Cherche une commande dans le système de commandes.

  Args:
    numero: le numéro de commande, CMD- suivi de 6 chiffres

  Returns:
    la fiche de la commande, ou une erreur si elle est introuvable
  """
  if not re.fullmatch(r"CMD-\d{6}", numero):
    return {"erreur": "numéro invalide : CMD- suivi de 6 chiffres"}
  if numero not in commandes.index:
    return {"erreur": f"commande introuvable : {numero}"}
  fiche = {"numero": numero, **commandes.loc[numero].to_dict()}
  fiche = {cle: valeur for cle, valeur in fiche.items() if ...}
  fiche["montant"] = float(fiche["montant"])
  fiche["paiements"] = ...
  return fiche


def verifier_stock(
  modele: Literal[tuple(MODELES)],
  taille: ...,
  magasin: ...,
) -> dict:
  """..."""
  ligne = stock[...]
  if ligne.empty:
    return {"erreur": "modèle, taille ou magasin inconnu"}
  quantite = int(ligne["quantite"].iloc[0])
  resultat = dict(modele=modele, taille=taille, magasin=magasin, quantite=quantite)
  if quantite == 0:
    resultat["reassort"] = ...
  return resultat


FONCTIONS = {
  f.__name__: f
  for f in [
    chercher_commande,
    verifier_stock,
    chercher_politique,
    creer_ticket,
    escalader,
  ]
}
OUTILS = [get_json_schema(f) for f in ...]
print("Schémas identiques à outils.json :", OUTILS == outils_reference)
lectures = ["chercher_commande", "verifier_stock", "chercher_politique"]
comparaisons = [
  FONCTIONS[a["name"]](**a["arguments"]) == r
  for appels, resultats in zip(
    demandes["appels_attendus"], demandes["resultats_outils"]
  )
  for a, r in zip(appels, resultats)
  if a["name"] in lectures
]
print(f"{sum(comparaisons)} résultats identiques sur {len(comparaisons)}")
```

### Solution

In [ ]:
def chercher_commande(numero: str) -> dict:
  """Cherche une commande dans le système de commandes.

  Args:
    numero: le numéro de commande, CMD- suivi de 6 chiffres

  Returns:
    la fiche de la commande, ou une erreur si elle est introuvable
  """
  if not re.fullmatch(r"CMD-\d{6}", numero):
    return {"erreur": "numéro invalide : CMD- suivi de 6 chiffres"}
  if numero not in commandes.index:
    return {"erreur": f"commande introuvable : {numero}"}
  fiche = {"numero": numero, **commandes.loc[numero].to_dict()}
  fiche = {cle: valeur for cle, valeur in fiche.items() if valeur != ""}
  fiche["montant"] = float(fiche["montant"])
  fiche["paiements"] = int(fiche["paiements"])
  return fiche


def verifier_stock(
  modele: Literal[tuple(MODELES)],
  taille: Literal[tuple(TAILLES)],
  magasin: Literal[tuple(MAGASINS)],
) -> dict:
  """Vérifie le stock d'un modèle de vélo dans un magasin.

  Args:
    modele: le modèle de vélo
    taille: la taille de cadre
    magasin: la ville du magasin

  Returns:
    la quantité en stock et, en rupture, la date du prochain réassort
  """
  ligne = stock[
    (stock["modele"] == modele)
    & (stock["taille"] == taille)
    & (stock["magasin"] == magasin)
  ]
  if ligne.empty:
    return {"erreur": "modèle, taille ou magasin inconnu"}
  quantite = int(ligne["quantite"].iloc[0])
  resultat = dict(modele=modele, taille=taille, magasin=magasin, quantite=quantite)
  if quantite == 0:
    resultat["reassort"] = ligne["reassort"].iloc[0]
  return resultat


FONCTIONS = {
  f.__name__: f
  for f in [
    chercher_commande,
    verifier_stock,
    chercher_politique,
    creer_ticket,
    escalader,
  ]
}
OUTILS = [get_json_schema(f) for f in FONCTIONS.values()]
print(json.dumps(OUTILS[1], ensure_ascii=False, indent=1))
print("Schémas identiques à outils.json :", OUTILS == outils_reference)

# Les trois outils de lecture rendent-ils ce que l'historique a enregistré ?
lectures = ["chercher_commande", "verifier_stock", "chercher_politique"]
comparaisons = [
  FONCTIONS[a["name"]](**a["arguments"]) == r
  for appels, resultats in zip(
    demandes["appels_attendus"], demandes["resultats_outils"]
  )
  for a, r in zip(appels, resultats)
  if a["name"] in lectures
]
print(f"{sum(comparaisons)} résultats identiques sur {len(comparaisons)}")

## 2. Le format conversationnel

Un exemple d'entraînement est une conversation entière : le message `system` (la consigne), le message du client (`user`), puis, pour chaque appel, un message `assistant` qui porte l'appel (`tool_calls`) suivi d'un message `tool` qui porte son résultat, et enfin la réponse de l'agent (`assistant`). C'est le format qu'attendent le *chat template* du modèle et `trl`.

Écrivez `vers_conversation(demande)`, qui fait cette conversation d'une demande de l'historique. Un appel s'écrit `{"type": "function", "function": {"name": ..., "arguments": ...}}`, ses arguments **en texte JSON** (`json.dumps`), comme le résultat, contenu du message `tool`. Puis passez au *chat template* de Qwen3 la conversation d'une demande à deux appels, avec les schémas (`tools=OUTILS`), et lisez le texte que voit le modèle : où vont les schémas, comment s'écrit un appel, quel rôle prend le résultat d'un outil ? Combien de tokens les outils ajoutent-ils à chaque requête ?

Pourquoi les arguments en texte ? Sur le disque, un jeu de données est une table Arrow, où une colonne n'a qu'un type : des arguments en dictionnaires, aux clés différentes d'un outil à l'autre, y deviendraient un seul dictionnaire de toutes les clés, les absentes à `null`, et le *chat template* les écrirait (`{"numero": "CMD-…", "modele": null, …}`). En texte, chaque appel garde les siens, et le *chat template* de Qwen3 les écrit tels quels.

**Documentation utile**

- [Les outils dans le *chat template*](https://huggingface.co/docs/transformers/en/chat_extras#tool-calling-example)
- [`apply_chat_template`](https://huggingface.co/docs/transformers/main_classes/tokenizer#transformers.PythonBackend.apply_chat_template)
- [`json.dumps`](https://docs.python.org/3/library/json.html#json.dumps)

In [ ]:
# Votre code ici

### Conseils supplémentaires

- Les messages s'écrivent `dict(role=..., content=...)` ; un appel : `dict(role="assistant", content="", tool_calls=[dict(type="function", function=dict(name=..., arguments=...))])`, puis son résultat : `dict(role="tool", content=json.dumps(resultat, ensure_ascii=False))`
- `zip(demande["appels_attendus"], demande["resultats_outils"])` : chaque appel avec son résultat
- `tokenizer.apply_chat_template(conversation, tools=OUTILS, tokenize=False)` : le texte ; `len(tokenizer(texte).input_ids)` : sa longueur en tokens, à comparer à celle du même texte sans `tools`

```python
def vers_conversation(demande) -> list[dict]:
  conversation = [
    dict(role="system", content=CONSIGNES_AGENT),
    dict(role="user", content=...),
  ]
  for appel, resultat in zip(demande["appels_attendus"], demande["resultats_outils"]):
    arguments = json.dumps(appel["arguments"], ensure_ascii=False)
    fonction = dict(name=appel["name"], arguments=arguments)
    conversation.append(
      dict(
        role="assistant", content="", tool_calls=[dict(type="function", function=...)]
      )
    )
    conversation.append(dict(role="tool", content=...))
  conversation.append(dict(role="assistant", content=demande["reponse"]))
  return conversation


exemple = demandes[demandes["appels_attendus"].str.len() == 2].iloc[0]
texte = tokenizer.apply_chat_template(
  vers_conversation(exemple), tools=..., tokenize=False
)
print(texte)
sans_outils = tokenizer.apply_chat_template(vers_conversation(exemple), tokenize=False)
print(len(tokenizer(texte).input_ids) - len(tokenizer(sans_outils).input_ids))
```

### Solution

In [ ]:
def vers_conversation(demande) -> list[dict]:
  conversation = [
    dict(role="system", content=CONSIGNES_AGENT),
    dict(role="user", content=demande["message"]),
  ]
  for appel, resultat in zip(demande["appels_attendus"], demande["resultats_outils"]):
    arguments = json.dumps(appel["arguments"], ensure_ascii=False)
    fonction = dict(name=appel["name"], arguments=arguments)
    conversation.append(
      dict(
        role="assistant",
        content="",
        tool_calls=[dict(type="function", function=fonction)],
      )
    )
    conversation.append(
      dict(role="tool", content=json.dumps(resultat, ensure_ascii=False))
    )
  conversation.append(dict(role="assistant", content=demande["reponse"]))
  return conversation


exemple = demandes[demandes["appels_attendus"].str.len() == 2].iloc[0]
texte = tokenizer.apply_chat_template(
  vers_conversation(exemple), tools=OUTILS, tokenize=False
)
print(texte)
sans_outils = tokenizer.apply_chat_template(vers_conversation(exemple), tokenize=False)
print(len(tokenizer(texte).input_ids), "tokens, dont", end=" ")
print(
  len(tokenizer(texte).input_ids) - len(tokenizer(sans_outils).input_ids),
  "pour les outils",
)

# Les schémas vont dans le message system, sous « # Tools » ; un appel est un bloc
# <tool_call> du message assistant ; le résultat d'un outil devient un message user,
# dans un bloc <tool_response>. Les cinq outils coûtent un millier de tokens à chaque
# requête, avant même le message du client. Seul le dernier message assistant commence
# par un bloc <think> vide : c'est le template de Qwen3 qui l'écrit, et le TP
# d'entraînement en tiendra compte.

## 3. Les erreurs de l'agent de référence

L'agent de référence, celui du chapitre précédent : le modèle de base, `MODELE_AGENT`, avec la consigne et les outils, dans sa boucle (`agent`, cellule ci-dessous). Il génère, on lit ses blocs `<tool_call>` (`lit_appels`), on exécute ses appels (`execute`), on lui rend leurs résultats, et il recommence tant qu'il appelle un outil. La cellule le fait tourner sur `echantillon`, `N_PAR_CLASSE` demandes de l'historique par comportement attendu (une seule sans GPU), jamais sur le jeu d'évaluation, qui reste figé pour comparer les modèles aux chapitres suivants : cinq minutes environ sur GPU, six sans. `trajets` garde, pour chaque demande, les appels obtenus et si leur format était lisible.

Relevez ses erreurs. Écrivez `valide(appel)`, qui renvoie la liste des problèmes d'un appel face à son schéma (`SCHEMAS`, par nom d'outil) : un outil inconnu, des arguments qui ne sont pas un objet, un argument requis manquant ou un argument inconnu, une valeur qui n'est pas une chaîne ou qui sort de son énumération, un numéro de commande mal formé ; une liste vide pour un appel valide. Puis `erreur_de(attendus, trajet)`, qui classe une demande, dans cet ordre :

1. `format` : une sortie illisible ;
2. `appel_manquant` : les outils obtenus sont le début des outils attendus (aucun appel, ou le premier seulement) ;
3. `appel_inutile` : l'inverse, des appels de trop, comme un appel là où il n'en fallait aucun ;
4. `mauvais_outil` : d'autres outils que les attendus ;
5. `arguments_invalides` : les bons outils, mais un appel que `valide` refuse ;
6. `arguments_faux` : des arguments valides mais pas les attendus, comparés sans les arguments libres (`ARGUMENTS_LIBRES` : `question` et `resume`, du texte qu'on ne compare pas mot à mot) ;
7. `correct`.

Rangez l'erreur de chaque demande dans la colonne `erreur` d'`echantillon`, ses appels obtenus dans `obtenus`, et croisez les erreurs avec les comportements attendus (`classe`). Quelles erreurs dominent, sur quels comportements ?

**Documentation utile**

- [`pandas.crosstab`](https://pandas.pydata.org/docs/reference/api/pandas.crosstab.html)

In [ ]:
BLOC_APPEL = re.compile(r"<tool_call>\s*(.*?)\s*</tool_call>", re.S)


def lit_appels(texte: str) -> list[dict] | None:
  """Les appels d'outil d'une sortie du modèle, None si un bloc est illisible."""
  appels = []
  for bloc in BLOC_APPEL.findall(texte):
    try:
      appel = json.loads(bloc)
      arguments = appel.get("arguments", {})
      if isinstance(arguments, str):
        arguments = json.loads(arguments)
    except (json.JSONDecodeError, AttributeError):
      return None
    if not isinstance(appel.get("name"), str):
      return None
    appels.append(dict(name=appel["name"], arguments=arguments))
  if texte.count("<tool_call>") != len(appels):  # un bloc pas refermé
    return None
  return appels


def execute(appel: dict) -> dict:
  """Le résultat d'un appel, ou l'erreur que l'agent recevrait."""
  if appel["name"] not in FONCTIONS:
    return {"erreur": f"outil inconnu : {appel['name']}"}
  try:
    return FONCTIONS[appel["name"]](**appel["arguments"])
  except (TypeError, KeyError, AttributeError):
    return {"erreur": "arguments invalides"}


def appel_chat(appel: dict) -> dict:
  """Un appel au format des conversations : ses arguments en texte JSON."""
  arguments = json.dumps(appel["arguments"], ensure_ascii=False)
  return dict(type="function", function=dict(name=appel["name"], arguments=arguments))


def charge(nom: str):
  return AutoModelForCausalLM.from_pretrained(nom, dtype=DTYPE, device_map=DEVICE)


def prefixe(modele, tok, systeme: str, outils: list | None = None):
  """Sans GPU, le cache du début commun des prompts (le message system et les
  outils), calculé une fois : chaque génération repart d'une copie, comme le fait un
  serveur d'inférence. Sur GPU, None : les lots vont assez vite sans."""
  if GPU:
    return None
  debut = tok.apply_chat_template(
    [dict(role="system", content=systeme)], tools=outils, tokenize=False
  )
  cache = DynamicCache(config=modele.config)
  with torch.no_grad():
    modele(**tok(debut, return_tensors="pt").to(DEVICE), past_key_values=cache)
  return cache


def genere_tout(
  modele, tok, conversations, outils=None, cache=None, max_tokens=96, **options
):
  """La suite de chaque conversation : par lots de 8 sur GPU, une à une sans GPU, à
  partir d'une copie de `cache`. Sans échantillonnage, sauf `options`."""
  options = options or dict(do_sample=False, temperature=None, top_p=None, top_k=None)
  prompts = [
    tok.apply_chat_template(
      c, tools=outils, tokenize=False, add_generation_prompt=True, enable_thinking=False
    )
    for c in conversations
  ]
  taille_lot = 8 if GPU else 1
  textes = []
  for debut in range(0, len(prompts), taille_lot):
    entrees = tok(
      prompts[debut : debut + taille_lot], return_tensors="pt", padding=True
    )
    copie = copy.deepcopy(cache) if cache is not None else None
    with torch.no_grad():
      sorties = modele.generate(
        **entrees.to(DEVICE),
        past_key_values=copie,
        max_new_tokens=max_tokens,
        **options,
      )
    textes += tok.batch_decode(
      sorties[:, entrees.input_ids.shape[1] :], skip_special_tokens=True
    )
  return textes


def agent(modele, messages: list[str], max_tours: int = 3) -> list[dict]:
  """La boucle de l'agent : génère, exécute les appels, recommence tant que le modèle
  appelle un outil. Pour chaque message : ses appels, et si son format était lisible.
  La réponse finale est coupée à 96 tokens : on ne regarde ici que les appels."""
  cache = prefixe(modele, tokenizer, CONSIGNES_AGENT, OUTILS)
  conversations = [
    [dict(role="system", content=CONSIGNES_AGENT), dict(role="user", content=m)]
    for m in messages
  ]
  trajets = [{"appels": [], "format_lisible": True, "texte": ""} for _ in messages]
  en_cours = list(range(len(messages)))
  for _ in range(max_tours):
    a_suivre = [conversations[i] for i in en_cours]
    textes = genere_tout(modele, tokenizer, a_suivre, outils=OUTILS, cache=cache)
    suite = []
    for i, texte in zip(en_cours, textes):
      appels = lit_appels(texte)
      trajets[i]["texte"] = texte
      if appels is None:
        trajets[i]["format_lisible"] = False
      elif appels:
        trajets[i]["appels"] += appels
        contenu = BLOC_APPEL.sub("", texte).strip()
        conversations[i].append(
          dict(
            role="assistant",
            content=contenu,
            tool_calls=[appel_chat(a) for a in appels],
          )
        )
        for a in appels:
          resultat = json.dumps(execute(a), ensure_ascii=False)
          conversations[i].append(dict(role="tool", content=resultat))
        suite.append(i)
    en_cours = suite
    if not en_cours:
      break
  return trajets


# Autant de demandes de chaque comportement attendu, au plus N_PAR_CLASSE.
echantillon = (
  demandes.sample(frac=1, random_state=0)
  .groupby("classe")
  .head(N_PAR_CLASSE)
  .reset_index(drop=True)
)
debut = time.time()
modele = charge(MODELE_AGENT)
trajets = agent(modele, list(echantillon["message"]))
print(f"{len(echantillon)} demandes en {time.time() - debut:.0f} s")
SCHEMAS = {outil["function"]["name"]: outil["function"] for outil in OUTILS}
ARGUMENTS_LIBRES = {"question", "resume"}

In [ ]:
# Votre code ici

### Conseils supplémentaires

- `SCHEMAS[nom]["parameters"]` : `properties`, chaque argument avec son `enum` éventuel, et `required`
- `proprietes[a].get("enum", [valeur])` : les valeurs permises, ou la valeur elle-même quand l'argument n'a pas d'énumération
- `noms_obtenus == noms_attendus[: len(noms_obtenus)]` : les outils obtenus sont le début des attendus (une liste vide aussi)
- `exacts(appel)` : les arguments d'un appel, sans les arguments libres
- Exemples à vérifier : `valide({"name": "verifier_stock", "arguments": {"modele": "Biclou Urbain 3", "taille": "XXL"}})` donne `['magasin manquant', 'taille hors de la liste : XXL']` ; `erreur_de([], {"appels": [], "format_lisible": True})` donne `'correct'`

```python
def valide(appel: dict) -> list[str]:
  schema = SCHEMAS.get(appel["name"])
  if schema is None:
    return [f"outil inconnu : {appel['name']}"]
  arguments = appel["arguments"]
  if not isinstance(arguments, dict):
    return ["les arguments ne sont pas un objet"]
  proprietes = schema["parameters"]["properties"]
  problemes = [f"{a} manquant" for a in schema["parameters"]["required"] if ...]
  for a, valeur in arguments.items():
    if a not in proprietes:
      problemes.append(f"{a} inconnu")
    elif not isinstance(valeur, str):
      problemes.append(f"{a} n'est pas une chaîne")
    elif valeur not in ...:
      problemes.append(f"{a} hors de la liste : {valeur}")
  numero = arguments.get("numero")
  if isinstance(numero, str) and not re.fullmatch(r"CMD-\d{6}", numero):
    problemes.append(f"numero mal formé : {numero}")
  return problemes


def exacts(appel: dict) -> dict:
  return {a: v for a, v in appel["arguments"].items() if a not in ARGUMENTS_LIBRES}


def erreur_de(attendus: list[dict], trajet: dict) -> str:
  obtenus = trajet["appels"]
  noms_attendus = [a["name"] for a in attendus]
  noms_obtenus = [a["name"] for a in obtenus]
  if not trajet["format_lisible"]:
    return "format"
  if noms_obtenus != noms_attendus:
    if noms_obtenus == noms_attendus[: len(noms_obtenus)]:
      return "appel_manquant"
    if ...:
      return "appel_inutile"
    return "mauvais_outil"
  if any(valide(a) for a in obtenus):
    return ...
  if any(exacts(o) != exacts(a) for o, a in zip(obtenus, attendus)):
    return "arguments_faux"
  return "correct"


echantillon["erreur"] = [
  erreur_de(attendus, t) for attendus, t in zip(echantillon["appels_attendus"], trajets)
]
echantillon["obtenus"] = [t["appels"] for t in trajets]
print(echantillon["erreur"].value_counts())
pd.crosstab(echantillon["classe"], echantillon["erreur"])
```

### Solution

In [ ]:
def valide(appel: dict) -> list[str]:
  schema = SCHEMAS.get(appel["name"])
  if schema is None:
    return [f"outil inconnu : {appel['name']}"]
  arguments = appel["arguments"]
  if not isinstance(arguments, dict):
    return ["les arguments ne sont pas un objet"]
  proprietes = schema["parameters"]["properties"]
  problemes = [
    f"{a} manquant" for a in schema["parameters"]["required"] if a not in arguments
  ]
  for a, valeur in arguments.items():
    if a not in proprietes:
      problemes.append(f"{a} inconnu")
    elif not isinstance(valeur, str):
      problemes.append(f"{a} n'est pas une chaîne")
    elif valeur not in proprietes[a].get("enum", [valeur]):
      problemes.append(f"{a} hors de la liste : {valeur}")
  numero = arguments.get("numero")
  if isinstance(numero, str) and not re.fullmatch(r"CMD-\d{6}", numero):
    problemes.append(f"numero mal formé : {numero}")
  return problemes


def exacts(appel: dict) -> dict:
  return {a: v for a, v in appel["arguments"].items() if a not in ARGUMENTS_LIBRES}


def erreur_de(attendus: list[dict], trajet: dict) -> str:
  obtenus = trajet["appels"]
  noms_attendus = [a["name"] for a in attendus]
  noms_obtenus = [a["name"] for a in obtenus]
  if not trajet["format_lisible"]:
    return "format"
  if noms_obtenus != noms_attendus:
    if noms_obtenus == noms_attendus[: len(noms_obtenus)]:
      return "appel_manquant"
    if noms_attendus == noms_obtenus[: len(noms_attendus)]:
      return "appel_inutile"
    return "mauvais_outil"
  if any(valide(a) for a in obtenus):
    return "arguments_invalides"
  if any(exacts(o) != exacts(a) for o, a in zip(obtenus, attendus)):
    return "arguments_faux"
  return "correct"


echantillon["erreur"] = [
  erreur_de(attendus, t) for attendus, t in zip(echantillon["appels_attendus"], trajets)
]
echantillon["obtenus"] = [t["appels"] for t in trajets]
print(echantillon["erreur"].value_counts())
for ligne in echantillon[echantillon["erreur"] != "correct"].head(3).itertuples():
  print(f"\n{ligne.erreur} ({ligne.scenario}) : {ligne.message[:80]}...")
  print("  attendus :", ligne.appels_attendus)
  print("  obtenus  :", ligne.obtenus)
pd.crosstab(echantillon["classe"], echantillon["erreur"])

# Le modèle de base répond souvent sans chercher : il écrit qu'il a vérifié le stock
# ou la commande sans appeler l'outil (appel_manquant). Qwen3-1.7B, sur GPU, appelle
# plus souvent, mais se trompe aussi d'outil ou d'arguments, et en appelle là où il
# fallait répondre. Ce sont ces demandes, et celles qui leur ressemblent, que le jeu
# doit couvrir d'abord.

## 4. Des erreurs aux exemples

Chaque erreur relevée désigne des exemples à mettre dans le jeu : la trajectoire correcte de la demande ratée, que l'historique a annotée, et d'autres demandes du même scénario, qui ont toutes les chances d'être ratées de la même façon. Construisez `depuis_erreurs` : les demandes de l'échantillon en erreur (colonne `source` : `"erreur"`), plus, pour chaque scénario en erreur, jusqu'à `N_VOISINS` demandes de l'historique hors de l'échantillon (`"voisin"`). Combien d'exemples, de quels comportements ? Regardez la conversation d'entraînement d'une demande ratée, face aux appels que l'agent avait faits.

**Documentation utile**

- [`Series.isin`](https://pandas.pydata.org/docs/reference/api/pandas.Series.isin.html)
- [`DataFrameGroupBy.head`](https://pandas.pydata.org/docs/reference/api/pandas.api.typing.DataFrameGroupBy.head.html)

In [ ]:
N_VOISINS = 10  # demandes de l'historique par scénario où l'agent s'est trompé

In [ ]:
# Votre code ici

### Conseils supplémentaires

- `demandes["scenario"].isin(echecs["scenario"])` : les demandes des scénarios en erreur ; `~demandes["id"].isin(echantillon["id"])` : hors de l'échantillon
- `.sample(frac=1, random_state=0).groupby("scenario").head(N_VOISINS)` : au plus `N_VOISINS` demandes au hasard par scénario
- `df.assign(source="erreur")` ajoute la colonne ; `pd.concat([...], ignore_index=True)` met les tableaux bout à bout

```python
echecs = echantillon[echantillon["erreur"] != "correct"]
voisins = (
  demandes[demandes["scenario"].isin(...) & ~demandes["id"].isin(echantillon["id"])]
  .sample(frac=1, random_state=0)
  .groupby("scenario")
  .head(...)
)
depuis_erreurs = pd.concat(
  [echecs.assign(source="erreur"), voisins.assign(source=...)], ignore_index=True
)
print(
  len(depuis_erreurs), "exemples :", depuis_erreurs["source"].value_counts().to_dict()
)
print(depuis_erreurs["classe"].value_counts())
if len(echecs):
  print("Obtenu :", echecs.iloc[0]["obtenus"])
  for message in vers_conversation(depuis_erreurs.iloc[0])[1:]:
    print(message)
```

### Solution

In [ ]:
echecs = echantillon[echantillon["erreur"] != "correct"]
voisins = (
  demandes[
    demandes["scenario"].isin(echecs["scenario"])
    & ~demandes["id"].isin(echantillon["id"])
  ]
  .sample(frac=1, random_state=0)
  .groupby("scenario")
  .head(N_VOISINS)
)
depuis_erreurs = pd.concat(
  [echecs.assign(source="erreur"), voisins.assign(source="voisin")], ignore_index=True
)
print(
  len(depuis_erreurs), "exemples :", depuis_erreurs["source"].value_counts().to_dict()
)
print(depuis_erreurs["classe"].value_counts())
if len(echecs):
  premier = depuis_erreurs.iloc[0]
  print("\nObtenu :", echecs.iloc[0]["obtenus"])
  for message in vers_conversation(premier)[1:]:
    print(message)

## 5. Des exemples synthétiques

Certains comportements manquent à l'historique : peu de demandes hors sujet ou à refuser, peu d'appels en série. Un modèle généraliste plus fort que l'agent, `MODELE_GENERALISTE`, peut écrire des demandes neuves. Dans ce sens-là, de l'appel au message, l'étiquette est connue d'avance : on fixe les appels, et le modèle écrit un message qui les demande.

La cellule ci-dessous compte, par comportement, ce qui manque à l'historique pour atteindre `CIBLE` exemples (`deficits` ; sans GPU, un exemple pour chacune des quatre plus grosses lacunes), puis tire les `cibles`. Chacune part d'une demande de l'historique du même comportement, sa graine, dont elle reprend les appels avec d'autres valeurs vérifiables (un numéro de commande qu'aucune demande ne cite, un autre vélo, une autre taille, un autre magasin) ; `infos` dit ce que le message devra citer tel quel, et `ton` le ton voulu.

Écrivez `demande_generation(cible)`, la conversation qui demande le message au généraliste : `GENERATION` en `system`, et en `user` la graine (`Exemple :`), le ton (`Ton :`), puis les informations à citer (`À citer tel quel :`), ou, quand il n'y en a pas, la consigne de ne citer aucun numéro de commande. Libérez l'agent (`del modele`, puis `libere()`), chargez le généraliste (`charge`) et son tokenizer, et générez un message par cible avec `genere_tout`, en échantillonnant (`do_sample=True`, `temperature=0.9`, `top_p=0.95`, 160 tokens au plus) ; rangez-le dans `cible["message"]`. Sans GPU, `prefixe` calcule une fois le début commun des prompts, la consigne `GENERATION`.

**Documentation utile**

- [`GenerationMixin.generate`](https://huggingface.co/docs/transformers/main_classes/text_generation#transformers.GenerationMixin.generate)

In [ ]:
CIBLE = 80  # exemples par comportement attendu visés dans le jeu final
deficits = (CIBLE - demandes["classe"].value_counts()).clip(lower=0)
deficits = deficits[deficits > 0].sort_values(ascending=False)
if N_SYNTHETIQUES:  # sans GPU : un exemple pour chacune des plus grosses lacunes
  deficits = pd.Series(1, index=deficits.index[:N_SYNTHETIQUES])
print(deficits.to_dict())

# Les commandes qu'aucune demande ne cite, ni l'historique ni l'évaluation : de quoi
# écrire des messages neufs. Le statut suit le scénario de l'exemple.
cites = set(
  re.findall(
    r"CMD-\d{6}", " ".join(demandes["message"]) + " ".join(eval_fige["message"])
  )
)
reserve = commandes[~commandes.index.isin(cites)]
STATUTS = {
  "colis_retard": "expediee",
  "colis_perdu": "expediee",
  "changer_adresse": "en_preparation",
  "remboursement_attente": "retour_en_controle",
}
TONS = [
  "poli et détaillé",
  "bref, sans formule de politesse",
  "agacé",
  "inquiet",
  "pressé, avec des fautes de frappe",
  "familier mais poli",
]


def cible_depuis(graine, hasard: random.Random) -> dict:
  """Une cible : les appels d'un exemple de l'historique (la graine), ses valeurs
  vérifiables remplacées par d'autres, et ce que le message devra en citer."""
  appels, infos = [], []
  numero = None
  for appel in graine["appels_attendus"]:
    arguments = dict(appel["arguments"])
    if appel["name"] == "chercher_commande":
      statut = STATUTS.get(graine["scenario"], "livree")
      numero = hasard.choice(sorted(reserve.index[reserve["statut"] == statut]))
      infos.append(f"le numéro de commande {numero}")
      arguments["numero"] = numero
    elif appel["name"] == "verifier_stock":
      arguments = dict(
        modele=hasard.choice(MODELES),
        taille=hasard.choice(TAILLES),
        magasin=hasard.choice(MAGASINS),
      )
      infos.append(
        f"le vélo {arguments['modele']}, en taille {arguments['taille']}, au magasin de {arguments['magasin']}"
      )
    elif "resume" in arguments and numero:
      arguments["resume"] = re.sub(r"CMD-\d{6}", numero, arguments["resume"])
    appels.append(dict(name=appel["name"], arguments=arguments))
  return {
    "classe": graine["classe"],
    "scenario": graine["scenario"],
    "graine": graine["message"],
    "ton": hasard.choice(TONS),
    "appels": appels,
    "infos": infos,
  }


hasard = random.Random(0)
cibles = [
  cible_depuis(
    demandes[demandes["classe"] == nom]
    .sample(1, random_state=hasard.randrange(10**6))
    .iloc[0],
    hasard,
  )
  for nom, n in deficits.items()
  for _ in range(n)
]
GENERATION = """Tu écris des messages de clients du service client de Biclou & Cie, magasins de vélos, pour entraîner son agent. On te donne un message d'exemple, le ton voulu et, parfois, des informations que le message doit citer telles quelles. Écris un seul nouveau message de client, en français, qui demande la même chose que l'exemple dans une autre situation : d'autres circonstances, d'autres détails, d'autres mots, aucune phrase de l'exemple. N'écris que le message : ni titre, ni guillemets, ni réponse."""
print(
  len(cibles),
  "cibles ; la première :",
  json.dumps(cibles[0], ensure_ascii=False, indent=1),
)

In [ ]:
# Votre code ici

### Conseils supplémentaires

- `genere_tout(generaliste, tok_generaliste, conversations, cache=prefixe(generaliste, tok_generaliste, GENERATION), max_tokens=160, do_sample=True, temperature=0.9, top_p=0.95)` : sans `outils`, le prompt n'a pas de schémas ; sur GPU, `prefixe` renvoie `None`
- `torch.manual_seed(0)` avant de générer : le même tirage à chaque exécution
- `tok_generaliste.padding_side = "left"` : pour générer par lots, les prompts sont complétés à gauche
- Sur GPU, cinq minutes environ pour 300 messages ; sans GPU, quatre minutes pour 4

```python
def demande_generation(cible: dict) -> list[dict]:
  consigne = f"Exemple :\n{cible['graine']}\n\nTon : {cible['ton']}\n"
  if cible["infos"]:
    consigne += "À citer tel quel : " + " ; ".join(...)
  else:
    consigne += "Le message ne cite aucun numéro de commande."
  return [dict(role="system", content=GENERATION), dict(role="user", content=...)]


debut = time.time()
del modele
libere()
generaliste = charge(...)
tok_generaliste = AutoTokenizer.from_pretrained(MODELE_GENERALISTE)
tok_generaliste.padding_side = "left"
torch.manual_seed(0)
textes = genere_tout(
  generaliste,
  tok_generaliste,
  [demande_generation(c) for c in cibles],
  cache=prefixe(generaliste, tok_generaliste, GENERATION),
  max_tokens=160,
  do_sample=...,
  temperature=0.9,
  top_p=0.95,
)
for cible, texte in zip(cibles, textes):
  cible["message"] = texte.strip()
print(f"{len(cibles)} messages en {time.time() - debut:.0f} s")
```

### Solution

In [ ]:
def demande_generation(cible: dict) -> list[dict]:
  consigne = f"Exemple :\n{cible['graine']}\n\nTon : {cible['ton']}\n"
  if cible["infos"]:
    consigne += "À citer tel quel : " + " ; ".join(cible["infos"])
  else:
    consigne += "Le message ne cite aucun numéro de commande."
  return [dict(role="system", content=GENERATION), dict(role="user", content=consigne)]


debut = time.time()
del modele
libere()
generaliste = charge(MODELE_GENERALISTE)
tok_generaliste = AutoTokenizer.from_pretrained(MODELE_GENERALISTE)
tok_generaliste.padding_side = "left"
torch.manual_seed(0)
textes = genere_tout(
  generaliste,
  tok_generaliste,
  [demande_generation(c) for c in cibles],
  cache=prefixe(generaliste, tok_generaliste, GENERATION),
  max_tokens=160,
  do_sample=True,
  temperature=0.9,
  top_p=0.95,
)
for cible, texte in zip(cibles, textes):
  cible["message"] = texte.strip()
print(f"{len(cibles)} messages en {time.time() - debut:.0f} s")
for cible in cibles[:3]:
  print(f"\n[{cible['classe']}] {cible['message']}")

## 6. Filtrer les exemples synthétiques

Un message généré n'est pas encore un exemple : il peut recopier son exemple, oublier le numéro ou en citer un autre, ou parler d'autre chose. APIGen (Liu et al., 2024) filtre ses exemples d'appels d'outils en trois étapes : le format, l'exécution des appels, puis le sens, vérifié par un modèle. Écrivez `raison_rejet(cible, gardes)`, qui renvoie pourquoi rejeter une cible, ou `None`, dans cet ordre :

1. `"format"` : le message fait moins de 3 mots ou plus de 150, contient un bloc `<tool_call>`, ou commence par un guillemet, `Exemple`, `Message` ou `Ton` ;
2. `"execution"` : un appel de la cible rend une erreur (`execute`) ; gardez les résultats dans `cible["resultats"]`, ce seront ceux de la conversation ;
3. `"coherence"` : le message ne cite pas un argument vérifiable de ses appels (`exacts`, de l'exercice 3 : le numéro, le modèle et le magasin, sans tenir compte des accents ni de la casse, et la taille, une lettre à chercher comme un mot), ou il cite un numéro de commande que ses appels n'ont pas, ou un numéro quand la cible n'a rien à citer ;
4. `"doublon"` : ses trigrammes de mots (`trigrammes`) ont un indice de Jaccard d'au moins `SEUIL_PROCHE` avec ceux d'un message de l'historique ou de l'évaluation (`deja_vus`), ou d'un message déjà gardé (`gardes`, que la fonction complète quand elle garde un message).

Appliquez-la à chaque cible, après avoir ôté les espaces et les guillemets autour de son message. Puis le contrôle du sens, en dernier car il coûte une génération : `verifie_sens` (cellule ci-dessous) fait classer par le généraliste chaque message encore gardé (un remerciement, une question hors sujet, une demande à refuser, ou une autre demande), et une cible dont le message n'a pas le type de son comportement est rejetée (`"sens"`). Comptez les rejets par raison, et lisez quelques messages gardés et rejetés : que laissent passer ces contrôles ?

**Documentation utile**

- [APIGen (Liu et al., 2024)](https://arxiv.org/abs/2406.18518)
- [`unicodedata.normalize`](https://docs.python.org/3/library/unicodedata.html#unicodedata.normalize)

In [ ]:
def trigrammes(texte: str) -> set[tuple]:
  mots = re.findall(r"[a-z0-9]+", unicodedata.normalize("NFKD", texte.lower()))
  return {tuple(mots[i : i + 3]) for i in range(max(1, len(mots) - 2))}


def jaccard(a: set, b: set) -> float:
  return len(a & b) / len(a | b) if a | b else 1.0


SEUIL_PROCHE = 0.6  # au-delà, deux messages sont presque les mêmes
deja_vus = [
  trigrammes(m) for m in pd.concat([demandes["message"], eval_fige["message"]])
]
TYPES = {
  "A": "un remerciement ou un accusé de réception, sans nouvelle demande",
  "B": "une question sans rapport avec le magasin, ses vélos ou ses services",
  "C": "une demande à refuser : les données d'un autre client, une fausse facture, une réduction exigée, un débridage, des instructions détournées",
  "D": "une demande au service client : une question, un problème, une commande à suivre",
}
TYPE_ATTENDU = {"aucun:remerciement": "A", "aucun:hors_sujet": "B", "aucun:refus": "C"}
JUGE_SENS = (
  "Tu classes les messages de clients du service client de Biclou & Cie, magasins de "
  "vélos. Réponds par la seule lettre du type du message, sans phrase :\n"
  + "\n".join(f"{lettre}. {texte}" for lettre, texte in TYPES.items())
)


def verifie_sens(cibles: list[dict]) -> list[bool]:
  """Le contrôle sémantique, par le modèle généraliste : le type de chaque message
  est-il celui de sa cible (D pour toutes celles qui appellent un outil, et pour un
  suivi de commande sans numéro) ?"""
  conversations = [
    [dict(role="system", content=JUGE_SENS), dict(role="user", content=c["message"])]
    for c in cibles
  ]
  cache = prefixe(generaliste, tok_generaliste, JUGE_SENS)
  textes = genere_tout(
    generaliste, tok_generaliste, conversations, cache=cache, max_tokens=10
  )
  lettres = [re.search(r"\b([ABCD])\b", texte) for texte in textes]
  return [
    bool(lettre) and lettre[1] == TYPE_ATTENDU.get(c["classe"], "D")
    for c, lettre in zip(cibles, lettres)
  ]

In [ ]:
# Votre code ici

### Conseils supplémentaires

- `unicodedata.normalize("NFKD", texte.lower())`, puis les caractères qui ne sont pas des accents (`not unicodedata.combining(c)`) : un texte comparable sans accents ni casse
- `re.search(rf"\b{valeur}\b", message)` : la taille comme un mot ; `re.findall(r"CMD-\d{6}", message)` : les numéros cités
- `any(jaccard(t, autre) >= SEUIL_PROCHE for autre in deja_vus + gardes)`, avec `t = trigrammes(message)`
- `verifie_sens(a_juger)` : un booléen par cible, vrai quand le type convient ; sans GPU, une minute pour 4 messages

```python
def sans_accents(texte: str) -> str:
  texte = unicodedata.normalize("NFKD", texte.lower())
  return "".join(c for c in texte if not unicodedata.combining(c))


def raison_rejet(cible: dict, gardes: list[set]) -> str | None:
  message = cible["message"]
  if not 3 <= len(message.split()) <= 150 or re.search(
    r"<tool_call>|^(exemple|message|ton)\b|^[\"«]", message, re.I
  ):
    return "format"
  cible["resultats"] = [execute(a) for a in cible["appels"]]
  if any("erreur" in r for r in ...):
    return "execution"
  texte = sans_accents(message)
  for appel in cible["appels"]:
    for nom, valeur in exacts(appel).items():
      if nom == "taille" and not re.search(rf"\b{valeur}\b", message):
        return "coherence"
      if nom in ("numero", "modele", "magasin") and ... not in texte:
        return "coherence"
  if not cible["infos"] and re.search(r"CMD-?\d", message, re.I):
    return "coherence"
  numeros = set(re.findall(r"CMD-\d{6}", message))
  if numeros - {a["arguments"].get("numero") for a in cible["appels"]}:
    return "coherence"
  t = trigrammes(message)
  if any(...):
    return "doublon"
  gardes.append(t)
  return None


gardes = []
for cible in cibles:
  cible["message"] = cible["message"].strip().strip('"«» ')
  cible["rejet"] = raison_rejet(cible, gardes)
a_juger = [c for c in cibles if c["rejet"] is None]
for cible, sens in zip(a_juger, verifie_sens(a_juger)):
  cible["rejet"] = None if sens else "sens"
print(pd.Series([c["rejet"] for c in cibles]).fillna("gardé").value_counts())
retenues = [c for c in cibles if c["rejet"] is None]
```

### Solution

In [ ]:
def sans_accents(texte: str) -> str:
  texte = unicodedata.normalize("NFKD", texte.lower())
  return "".join(c for c in texte if not unicodedata.combining(c))


def raison_rejet(cible: dict, gardes: list[set]) -> str | None:
  message = cible["message"]
  # 1. Le format : un message de client, rien d'autre.
  if not 3 <= len(message.split()) <= 150 or re.search(
    r"<tool_call>|^(exemple|message|ton)\b|^[\"«]", message, re.I
  ):
    return "format"
  # 2. L'exécution : les appels cibles passent sur les données du magasin.
  cible["resultats"] = [execute(a) for a in cible["appels"]]
  if any("erreur" in r for r in cible["resultats"]):
    return "execution"
  # 3. La cohérence : le message cite les arguments vérifiables, et rien d'autre.
  texte = sans_accents(message)
  for appel in cible["appels"]:
    for nom, valeur in exacts(appel).items():
      if nom == "taille" and not re.search(rf"\b{valeur}\b", message):
        return "coherence"
      if nom in ("numero", "modele", "magasin") and sans_accents(valeur) not in texte:
        return "coherence"
  if not cible["infos"] and re.search(r"CMD-?\d", message, re.I):
    return "coherence"
  numeros = set(re.findall(r"CMD-\d{6}", message))
  if numeros - {a["arguments"].get("numero") for a in cible["appels"]}:
    return "coherence"
  # 4. Les doublons : exacts ou proches, de l'historique, de l'évaluation, ou entre eux.
  t = trigrammes(message)
  if any(jaccard(t, autre) >= SEUIL_PROCHE for autre in deja_vus + gardes):
    return "doublon"
  gardes.append(t)
  return None


gardes = []
for cible in cibles:
  cible["message"] = cible["message"].strip().strip('"«» ')
  cible["rejet"] = raison_rejet(cible, gardes)
# 5. Le sens, en dernier car il coûte une génération : le juge, sur ce qui reste.
a_juger = [c for c in cibles if c["rejet"] is None]
for cible, sens in zip(a_juger, verifie_sens(a_juger)):
  cible["rejet"] = None if sens else "sens"
rejets = pd.Series([c["rejet"] for c in cibles]).fillna("gardé")
print(rejets.value_counts())
for cible in cibles[:4]:
  print(f"\n[{cible['rejet'] or 'gardé'}] {cible['classe']} : {cible['message']}")
retenues = [c for c in cibles if c["rejet"] is None]

# Les règles ne voient que la forme et les valeurs : un message qui parle d'autre
# chose que sa cible, une question de livraison écrite pour un remerciement, ne se voit
# qu'au contrôle du sens, et un petit juge en laisse passer. Le reste, seule une
# relecture le dit : celle d'un échantillon des exemples gardés, avant de valider le
# jeu.

## 7. Équilibrer, découper, enregistrer

La cellule ci-dessous écrit la réponse finale des exemples synthétiques gardés : le généraliste y tient le rôle de l'agent, avec la même consigne et les mêmes outils, sur la trajectoire aux appels fixés, et les réponses qui ne suivent pas la forme de la charte (`respecte_charte`, des TP précédents) sont écartées. `synthetiques` a les colonnes de `demandes` ; sans GPU, comptez cinq minutes.

Puis constituez le jeu final, en quatre temps :

1. Équilibrer : les exemples tirés des erreurs (`depuis_erreurs`), les synthétiques, puis l'historique, jusqu'à `CIBLE` exemples par comportement (`classe`), sans message en double. Comparez les effectifs par comportement dans l'historique et dans le jeu, la part des demandes sans outil, et celle des demandes à plusieurs appels.
2. Écarter ce qui approche le jeu d'évaluation figé : un message à `SEUIL_PROCHE` ou plus d'un message d'évaluation, ou qui cite l'un de ses numéros de commande.
3. Découper en entraînement et validation (10 %) par formulation : les messages d'une même formulation de l'historique (`gabarit`) restent du même côté, sans quoi la validation mesurerait des messages que l'entraînement a vus à quelques mots près. Vérifiez qu'aucun gabarit n'est des deux côtés.
4. Enregistrer chaque partie en JSON Lines, `agent-sft-train.jsonl` et `agent-sft-validation.jsonl`, au format conversationnel de `trl` : une colonne `messages` (`vers_conversation`) et une colonne `tools`, les schémas en texte JSON (le `SFTTrainer` de `trl` les relit, en texte comme en liste), plus `id`, `source` et `classe`. Rechargez-les avec `load_dataset` et repassez un exemple au *chat template*. Enfin, si vous avez un compte Hugging Face, publiez le jeu sur le Hub, en **privé**, étiqueté `v1.0`, avec le jeton du secret Colab `HF_TOKEN`, comme au TP de préparation des données ; sans lui, la cellule ne publie rien.

**Documentation utile**

- [`DataFrameGroupBy.cumcount`](https://pandas.pydata.org/docs/reference/api/pandas.api.typing.DataFrameGroupBy.cumcount.html)
- [`GroupShuffleSplit`](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.GroupShuffleSplit.html)
- [Le format de `trl` avec des outils](https://huggingface.co/docs/trl/dataset_formats#tool-calling)
- [`load_dataset`](https://huggingface.co/docs/datasets/package_reference/loading_methods#datasets.load_dataset)
- [`DatasetDict.push_to_hub`](https://huggingface.co/docs/datasets/package_reference/main_classes#datasets.DatasetDict.push_to_hub)

In [ ]:
SIGNATURE = "Bonne journée,\nL'équipe Biclou & Cie"
TUTOIEMENT = re.compile(r"\b(tu|te|toi|ton|ta|tes)\b|\bt['’]", re.IGNORECASE)


def respecte_charte(reponse: str) -> bool:
  """Les règles de forme de la charte, comme dans les TP précédents."""
  reponse = reponse.strip()
  return (
    reponse.startswith(("Bonjour", "Hello"))
    and reponse.endswith((SIGNATURE, "The Biclou & Cie team"))
    and not TUTOIEMENT.search(reponse)
    and 40 <= len(reponse.split()) <= 120
  )


# La réponse finale de chaque exemple synthétique : le modèle généraliste tient le rôle
# de l'agent, sur la trajectoire aux appels fixés, et la charte trie ses réponses.
debut = time.time()
lignes = [
  dict(
    id=f"S-{i:03d}",
    message=c["message"],
    scenario=c["scenario"],
    gabarit=f"synthetique/{i:03d}",
    classe=c["classe"],
    appels_attendus=c["appels"],
    resultats_outils=c["resultats"],
    reponse="",
  )
  for i, c in enumerate(retenues, 1)
]
reponses = genere_tout(
  generaliste,
  tok_generaliste,
  [vers_conversation(ligne)[:-1] for ligne in lignes],
  outils=OUTILS,
  cache=prefixe(generaliste, tok_generaliste, CONSIGNES_AGENT, OUTILS),
  max_tokens=180,
)
for ligne, reponse in zip(lignes, reponses):
  # Sans les espaces de fin de ligne que le modèle laisse parfois (« Bonne journée,  »).
  ligne["reponse"] = "\n".join(l.rstrip() for l in reponse.strip().splitlines())
synthetiques = pd.DataFrame(
  [
    l
    for l in lignes
    if "<tool_call>" not in l["reponse"] and respecte_charte(l["reponse"])
  ],
  columns=list(lignes[0]) if lignes else ["id", "classe", "gabarit", "message"],
).assign(source="synthetique")
print(
  f"{len(synthetiques)} exemples synthétiques sur {len(lignes)}, en {time.time() - debut:.0f} s"
)
if len(synthetiques):
  print(synthetiques.iloc[0]["message"], "\n\n", synthetiques.iloc[0]["reponse"])

In [ ]:
# Votre code ici

### Conseils supplémentaires

- `pd.concat([...]).drop_duplicates("message")` garde la première occurrence : mettez les sources dans l'ordre de priorité
- `candidats.groupby("classe").cumcount()` : le rang de chaque exemple dans son comportement ; gardez ceux de rang inférieur à `CIBLE`, et tous ceux qui ne viennent pas de l'historique
- `GroupShuffleSplit(n_splits=1, test_size=0.1, random_state=0).split(jeu, groups=jeu["gabarit"])` : un itérateur de paires d'indices (entraînement, validation), d'où `next(...)`
- Une ligne du fichier : `json.dumps(dict(id=..., source=..., classe=..., messages=vers_conversation(d), tools=json.dumps(OUTILS, ensure_ascii=False)), ensure_ascii=False)`
- `load_dataset("json", data_files={"train": ..., "validation": ...})` ; `json.loads(exemple["tools"])` rend les schémas au *chat template*
- La publication, comme au TP de préparation des données : `jeux.push_to_hub(depot, private=True, token=jeton)` et `HfApi(token=jeton).create_tag(depot, tag="v1.0", repo_type="dataset", exist_ok=True)`

```python
from sklearn.model_selection import GroupShuffleSplit

historique = demandes.sample(frac=1, random_state=0).assign(source="historique")
candidats = pd.concat(
  [depuis_erreurs, synthetiques, historique], ignore_index=True
).drop_duplicates("message")
rang = candidats.groupby("classe").cumcount()
jeu = candidats[(rang < CIBLE) | (candidats["source"] != "historique")]
jeu = jeu.reset_index(drop=True)
avant_apres = pd.DataFrame(
  {"historique": demandes["classe"].value_counts(), "jeu": jeu["classe"].value_counts()}
)
print(avant_apres.fillna(0).astype(int))

evaluation = [trigrammes(m) for m in eval_fige["message"]]
numeros_eval = set(re.findall(r"CMD-\d{6}", " ".join(eval_fige["message"])))
proches = jeu["message"].map(
  lambda m: (
    max(jaccard(trigrammes(m), e) for e in evaluation) >= SEUIL_PROCHE
    or bool(set(re.findall(r"CMD-\d{6}", m)) & ...)
  )
)
jeu = jeu[~proches].reset_index(drop=True)

decoupe = GroupShuffleSplit(n_splits=1, test_size=0.1, random_state=0)
indices_train, indices_validation = next(decoupe.split(jeu, groups=...))
parties = {"train": jeu.iloc[indices_train], "validation": jeu.iloc[indices_validation]}
print(set(parties["train"]["gabarit"]) & set(parties["validation"]["gabarit"]))

outils_json = json.dumps(OUTILS, ensure_ascii=False)
for nom, partie in parties.items():
  with open(f"agent-sft-{nom}.jsonl", "w", encoding="utf-8") as f:
    for d in partie.to_dict("records"):
      ligne = dict(
        id=d["id"],
        source=d["source"],
        classe=d["classe"],
        messages=...,
        tools=outils_json,
      )
      f.write(json.dumps(ligne, ensure_ascii=False) + "\n")
jeux = load_dataset(
  "json", data_files={nom: f"agent-sft-{nom}.jsonl" for nom in parties}
)
exemple = jeux["train"][0]
print(tokenizer.apply_chat_template(exemple["messages"], tools=..., tokenize=False))
```

### Solution

In [ ]:
from sklearn.model_selection import GroupShuffleSplit

# 1. Équilibrer : les exemples des erreurs, les synthétiques, puis l'historique,
# jusqu'à CIBLE exemples par comportement.
candidats = pd.concat(
  [
    depuis_erreurs,
    synthetiques,
    demandes.sample(frac=1, random_state=0).assign(source="historique"),
  ],
  ignore_index=True,
).drop_duplicates("message")
rang = candidats.groupby("classe").cumcount()
jeu = candidats[(rang < CIBLE) | (candidats["source"] != "historique")].reset_index(
  drop=True
)
avant_apres = (
  pd.DataFrame(
    {
      "historique": demandes["classe"].value_counts(),
      "jeu": jeu["classe"].value_counts(),
    }
  )
  .fillna(0)
  .astype(int)
)
print(avant_apres.sort_values("jeu", ascending=False))
print(
  "sans outil :",
  f"{(demandes['classe'].str.startswith('aucun')).mean():.0%}",
  "->",
  f"{(jeu['classe'].str.startswith('aucun')).mean():.0%}",
)
print(
  "plusieurs appels :",
  f"{(demandes['appels_attendus'].str.len() > 1).mean():.0%}",
  "->",
  f"{(jeu['appels_attendus'].str.len() > 1).mean():.0%}",
)
print(jeu["source"].value_counts().to_dict())

# 2. Rien près de l'évaluation figée : ni message proche, ni numéro de commande.
evaluation = [trigrammes(m) for m in eval_fige["message"]]
numeros_eval = set(re.findall(r"CMD-\d{6}", " ".join(eval_fige["message"])))
proches = jeu["message"].map(
  lambda m: (
    max(jaccard(trigrammes(m), e) for e in evaluation) >= SEUIL_PROCHE
    or bool(set(re.findall(r"CMD-\d{6}", m)) & numeros_eval)
  )
)
print(proches.sum(), "exemples trop proches de l'évaluation, retirés")
jeu = jeu[~proches].reset_index(drop=True)

# 3. Découper par formulation : aucun gabarit des deux côtés.
decoupe = GroupShuffleSplit(n_splits=1, test_size=0.1, random_state=0)
indices_train, indices_validation = next(decoupe.split(jeu, groups=jeu["gabarit"]))
parties = {"train": jeu.iloc[indices_train], "validation": jeu.iloc[indices_validation]}
communs = set(parties["train"]["gabarit"]) & set(parties["validation"]["gabarit"])
print({nom: len(p) for nom, p in parties.items()}, "; gabarits communs :", len(communs))

# 4. Enregistrer au format conversationnel de trl, avec la colonne tools.
outils_json = json.dumps(OUTILS, ensure_ascii=False)
for nom, partie in parties.items():
  lignes_trl = [
    dict(
      id=d["id"],
      source=d["source"],
      classe=d["classe"],
      messages=vers_conversation(d),
      tools=outils_json,
    )
    for d in partie.to_dict("records")
  ]
  with open(f"agent-sft-{nom}.jsonl", "w", encoding="utf-8") as f:
    for ligne in lignes_trl:
      f.write(json.dumps(ligne, ensure_ascii=False) + "\n")
jeux = load_dataset(
  "json",
  data_files={nom: f"agent-sft-{nom}.jsonl" for nom in parties},
)
print(jeux)
exemple = jeux["train"][0]
print(
  tokenizer.apply_chat_template(
    exemple["messages"], tools=json.loads(exemple["tools"]), tokenize=False
  )[-600:]
)

jeton = os.environ.get("HF_TOKEN")
try:
  from google.colab import userdata

  jeton = userdata.get("HF_TOKEN")
except Exception:
  pass
if jeton:
  from huggingface_hub import HfApi

  api = HfApi(token=jeton)
  depot = f"{api.whoami()['name']}/agent-sft"
  jeux.push_to_hub(depot, private=True, token=jeton)
  api.create_tag(depot, tag="v1.0", repo_type="dataset", exist_ok=True)
  print("Publié :", depot)
else:
  print("Pas de jeton HF_TOKEN : rien n'est publié")

# L'historique donnait 43 % de recherches dans les règles et 1 % de demandes à
# refuser ; le jeu s'en tient à CIBLE exemples par comportement, ceux des erreurs
# d'abord. Sans GPU, les cas rares restent rares : c'est le jeu de référence, construit
# sur GPU, que charge le TP suivant.

## À retenir

- Un outil est un schéma JSON, tiré d'une fonction Python typée et documentée : son nom, sa description et ses énumérations sont ce que le modèle voit, et ce qu'on vérifie de ses appels.
- Un exemple d'appel d'outil est une conversation entière : les appels, leurs résultats, la réponse. Les arguments s'y écrivent en texte JSON, pour que le jeu reste une table.
- Les erreurs de l'agent de référence disent quels exemples ajouter d'abord : la demande ratée, annotée, et ses semblables.
- De l'appel au message, l'étiquette d'un exemple synthétique est connue d'avance ; reste à vérifier que le message la mérite : le format, l'exécution, la cohérence, les doublons, puis le sens, par un modèle, et une relecture.
- Le jeu s'équilibre par comportement, cas sans outil compris, se découpe par formulation, et se tient loin du jeu d'évaluation figé.